# Data Preparation: Value Transformations with GNIS & Baseball Examples

In [1]:
import numpy as np
import pandas as pd

Only run the following if you need to recreate the baseball database:

In [2]:
!psql -h localhost -c 'DROP DATABASE IF EXISTS baseball'
!psql -h localhost -c 'CREATE DATABASE baseball'
!psql -h localhost -d baseball -f data/baseball.sql

DROP DATABASE
CREATE DATABASE
SET
SET
SET
SET
SET
 set_config 
------------
 
(1 row)

SET
SET
SET
SET
SET
SET
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
CREATE TABLE
COPY 5219
COPY 104256
COPY 179
COPY 6236
COPY 425
COPY 6879
COPY 104324
COPY 13943
COPY 17350
COPY 138838
COPY 12028
COPY 31955
COPY 13110
COPY 4191
COPY 3040
COPY 3469
COPY 93
COPY 252
COPY 19370
COPY 45806
COPY 5445
COPY 26428
COPY 1207
COPY 325
COPY 2865
COPY 120
COPY 52
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE
ALTER TABLE

In [3]:
## we'll use the Lahman baseball database in our examples today.
%reload_ext sql
%sql postgresql://localhost:5432/baseball
%config SqlMagic.displaylimit = 20

Connecting to 'postgresql://localhost:5432/baseball'

---
# Scalar and Window Function Demos: Baseball database
## [At Home, Slides in Class] Scalar Functions and Query Plans

Let's analyze the number of scalar functions in the following query:

In [4]:
%%sql
WITH year_num AS
  (SELECT year_id, (year_id % 100) as year
     FROM batting
  )
SELECT year_id, CONCAT('''', LPAD(year::text, 2, '0')) as year
  FROM year_num
 LIMIT 5;

Running query in 'postgresql://localhost:5432/baseball'

5 rows affected.

year_id,year
1871,'71
1871,'71
1871,'71
1871,'71
1871,'71


We can analyze it by looking at the EXPLAIN with VERBOSE (we've flattened it for convenience):

In [5]:
%%sql
EXPLAIN (VERBOSE true)
SELECT year_id,
       CONCAT('''', LPAD((year_id % 100)::text, 2, '0')) AS year
FROM batting;


Running query in 'postgresql://localhost:5432/baseball'

2 rows affected.

QUERY PLAN
Seq Scan on public.batting (cost=0.00..3927.29 rows=104324 width=36)
"Output: year_id, concat('''', lpad(((year_id % 100))::text, 2, '0'::text))"


What if scalar functions mention multiple tables?

The below query computes an arbitrary statistic for pitchers:
* 1 point for every strikeout they throw as pitcher
* –1 for every point they themselves struck out as batter

If the notebook-like output is hard to read, try out the query in `psql`. Note that notebooks don't preserve whitespace when displaying dataframes.

In [6]:
%%sql
EXPLAIN (VERBOSE true)
SELECT p.player_id, p.so - b.so
  FROM pitching p
  INNER JOIN batting b
  ON p.player_id=b.player_id;

Running query in 'postgresql://localhost:5432/baseball'

11 rows affected.

QUERY PLAN
Nested Loop (cost=0.43..13003.79 rows=341428 width=13)
"Output: p.player_id, (p.so - b.so)"
-> Seq Scan on public.pitching p (cost=0.00..1374.06 rows=45806 width=13)
"Output: p.player_id, p.year_id, p.stint, p.team_id, p.lg_id, p.w, p.l, p.g, p.gs, p.cg, p.sho, p.sv, p.ipouts, p.h, p.er, p.hr, p.bb, p.so, p.baopp, p.era, p.ibb, p.wp, p.hbp, p.bk, p.bfp, p.gf, p.r, p.sh, p.sf, p.gidp"
-> Memoize (cost=0.43..0.73 rows=7 width=13)
"Output: b.so, b.player_id"
Cache Key: p.player_id
Cache Mode: logical
-> Index Scan using batting_pkey on public.batting b (cost=0.42..0.72 rows=7 width=13)
"Output: b.so, b.player_id"


---
## Window Functions [In Class if Time Permits]

Window functions, without partitions first, and then with partitions. <br/>


In [7]:
%%sql
SELECT name_first, name_last, year_id, HR,
       rank() OVER (ORDER BY HR DESC)
       -- , avg(HR)    OVER (PARTITION BY b.player_id ORDER BY year_id ROWS 3 PRECEDING) as avg_4yr
       -- , lag(HR, 7) OVER (PARTITION BY b.player_id ORDER BY year_id) as previous
       -- , lag(HR, 2) OVER (PARTITION BY b.player_id ORDER BY year_id) as lag2
FROM batting b, people p
WHERE p.player_id = b.player_id
   AND (name_last = 'Bonds' or name_last = 'Ruth')
ORDER BY HR DESC
LIMIT 10;

Running query in 'postgresql://localhost:5432/baseball'

10 rows affected.

name_first,name_last,year_id,hr,rank
Barry,Bonds,2001,73,1
Babe,Ruth,1927,60,2
Babe,Ruth,1921,59,3
Babe,Ruth,1920,54,4
Babe,Ruth,1928,54,4
Barry,Bonds,2000,49,6
Babe,Ruth,1930,49,6
Babe,Ruth,1926,47,8
Barry,Bonds,1993,46,9
Barry,Bonds,2002,46,9


Same query, different order by - so that we can inspect the other attributes
<br/>
Note ROWS 3 PRECEDING = RANGE BETWEEN 3 PRECEDING AND CURRENT ROW

In [8]:
%%sql
SELECT name_first, name_last, year_id, HR
       -- rank() OVER (ORDER BY HR DESC)
       , avg(HR)    OVER (PARTITION BY b.player_id ORDER BY year_id ROWS 3 PRECEDING) as avg_4yr
       , lag(HR, 1) OVER (PARTITION BY b.player_id ORDER BY year_id) as previous
       , lag(HR, 2) OVER (PARTITION BY b.player_id ORDER BY year_id) as lag2
FROM batting b, people p
WHERE p.player_id = b.player_id
   AND (name_last = 'Bonds' or name_last = 'Ruth')
ORDER BY b.player_id, year_id 
LIMIT 20;

Running query in 'postgresql://localhost:5432/baseball'

20 rows affected.

name_first,name_last,year_id,hr,avg_4yr,previous,lag2
Barry,Bonds,1986,16,16.0000000000000000,None,None
Barry,Bonds,1987,25,20.5000000000000000,16,None
Barry,Bonds,1988,24,21.6666666666666667,25,16
Barry,Bonds,1989,19,21.0000000000000000,24,25
Barry,Bonds,1990,33,25.2500000000000000,19,24
Barry,Bonds,1991,25,25.2500000000000000,33,19
Barry,Bonds,1992,34,27.7500000000000000,25,33
Barry,Bonds,1993,46,34.5000000000000000,34,25
Barry,Bonds,1994,37,35.5000000000000000,46,34
Barry,Bonds,1995,33,37.5000000000000000,37,46


---
## [At Home] Inverse Distribution Window Functions

In [9]:
%%sql
SELECT MIN(HR),
       percentile_cont(0.25) WITHIN GROUP (ORDER BY HR) AS p25,
       percentile_cont(0.50) WITHIN GROUP (ORDER BY HR) AS median,
       percentile_cont(0.75) WITHIN GROUP (ORDER BY HR) AS p75,
       percentile_cont(0.99) WITHIN GROUP (ORDER BY HR) AS p99,
       MAX(HR),
       AVG(HR) AS "average hit rate"
FROM batting;

Running query in 'postgresql://localhost:5432/baseball'

1 rows affected.

min,p25,median,p75,p99,max,average hit rate
0,0.0,0.0,2.0,31.0,73,2.8315823779763046


Now just to verify that the max is 73, we can run the following query:

In [10]:
%%sql
SELECT HR, COUNT(*) FROM batting GROUP BY HR ORDER BY HR DESC;

Running query in 'postgresql://localhost:5432/baseball'

67 rows affected.

hr,count
73,1
70,1
66,1
65,1
64,1
63,1
61,1
60,1
59,2
58,3


---
### [At Home, Slides in Class] Hypothetical-Set Window Functions

In [11]:
hrs = 70 # hypothetically, four home runs

In [12]:
%%sql
SELECT {{hrs}} as hypothetical,
       rank({{hrs}}) WITHIN GROUP (ORDER BY HR DESC),
       dense_rank({{hrs}}) WITHIN GROUP (ORDER BY HR DESC),
       percent_rank({{hrs}}) WITHIN GROUP (ORDER BY HR DESC) * 100 AS pct_rank,
       cume_dist({{hrs}}) WITHIN GROUP (ORDER BY HR)
FROM batting;

Running query in 'postgresql://localhost:5432/baseball'

1 rows affected.

hypothetical,rank,dense_rank,pct_rank,cume_dist
70,2,2,0.000958552202752962,0.9999904145698538


Without `jupysql` variable substituion

In [13]:
%%sql
SELECT 4 as hypothetical,
       rank(4) WITHIN GROUP (ORDER BY HR DESC),
       dense_rank(4) WITHIN GROUP (ORDER BY HR DESC),
       percent_rank(4) WITHIN GROUP (ORDER BY HR DESC) * 100 AS pct_rank,
       cume_dist(4) WITHIN GROUP (ORDER BY HR)
FROM batting;

Running query in 'postgresql://localhost:5432/baseball'

1 rows affected.

hypothetical,rank,dense_rank,pct_rank,cume_dist
4,18420,63,17.655573022506807,0.823445962137551


# Granularity Demos: GNIS

This notebook transforms the existing [Geographics Names Information Systems (GNIS)](https://www.usgs.gov/core-science-systems/ngp/board-on-geographic-names/download-gnis-data) national zip file.

We have provided a subset of the sql database for you in `data/national.sql`.



In [14]:
!psql -h localhost -d gnis -c 'SELECT pg_terminate_backend(pg_stat_activity.pid) FROM pg_stat_activity WHERE datname = current_database() AND pid <> pg_backend_pid();'
!psql -h localhost -c 'DROP DATABASE IF EXISTS gnis'
!psql -h localhost -c 'CREATE DATABASE gnis' 
!psql -h localhost -d gnis -f data/gnis.sql

 pg_terminate_backend 
----------------------
(0 rows)

DROP DATABASE
CREATE DATABASE
SET
SET
SET
SET
SET
 set_config 
------------
 
(1 row)

SET
SET
SET
SET
SET
SET
CREATE TABLE
ALTER TABLE
CREATE TABLE
ALTER TABLE
COPY 3195
COPY 11533
CREATE INDEX


In [15]:
%reload_ext sql
%sql postgresql://localhost:5432/gnis
%config SqlMagic.displaylimit = 15

Connecting and switching to connection 'postgresql://localhost:5432/gnis'

* View schema in `psql`
* View some rows below

In [16]:
%sql SELECT COUNT(*) FROM national;

Running query in 'postgresql://localhost:5432/gnis'

1 rows affected.

count
11533


In [17]:
%sql SELECT * FROM national WHERE county_name = 'Alameda';

Running query in 'postgresql://localhost:5432/gnis'

21 rows affected.

feature_id,feature_name,feature_class,state_alpha,state_numeric,county_name,county_numeric,primary_lat_dms,prim_long_dms,prim_lat_dec,prim_long_dec,source_lat_dms,source_long_dms,source_lat_dec,source_long_dec,elev_in_m,elev_in_ft,map_name,date_created,date_edited
218316,Apperson Creek,Stream,CA,6,Alameda,1.0,373349N,1215000W,37.5635453,-121.8332887,373232N,1214804W,37.5422222,-121.8011111,148.0,486.0,La Costa Valley,01/19/1981,None
225998,Irvington High School,School,CA,6,Alameda,1.0,373126N,1215801W,37.523814,-121.9670659,None,None,None,None,13.0,43.0,Niles,01/19/1981,03/31/2021
226951,Laurel Elementary School,School,CA,6,Alameda,1.0,374734N,1221147W,37.792899,-122.1964288,None,None,None,None,68.0,223.0,Oakland East,06/14/2000,03/14/2021
229367,Murray Elementary School,School,CA,6,Alameda,1.0,374318N,1215557W,37.721801,-121.9326269,None,None,None,None,112.0,367.0,Dublin,01/19/1981,03/14/2021
235581,Strawberry Creek,Stream,CA,6,Alameda,1.0,375221N,1221443W,37.8724258,-122.2452464,375251N,1221354W,37.8807588,-122.2316349,154.0,505.0,Oakland East,01/19/1981,08/31/2016
1654274,Hayward Golf Course,Locale,CA,6,Alameda,1.0,373726N,1220250W,37.6238222,-122.0471843,None,None,None,None,5.0,16.0,Newark,01/19/1981,None
1664964,KOFY-AM (San Mateo),Tower,CA,6,Alameda,1.0,374934N,1221842W,37.8260385,-122.3116366,None,None,None,None,2.0,7.0,Oakland West,07/01/1994,None
1670278,Lake Elizabeth,Lake,CA,6,Alameda,1.0,373255N,1215742W,37.5487056,-121.9617554,None,None,None,None,16.0,52.0,Niles,11/09/1995,03/07/2019
1692819,California School for the Deaf - Fremont,School,CA,6,Alameda,1.0,373334N,1215747W,37.5593966,-121.9631843,None,None,None,None,20.0,66.0,Niles,05/08/1996,09/16/2016
1692863,J A Freitas Library,Building,CA,6,Alameda,1.0,374335N,1220925W,37.7263185,-122.1569101,None,None,None,None,19.0,62.0,San Leandro,05/08/1996,None


In [18]:
%%sql
SELECT *
FROM national TABLESAMPLE BERNOULLI(10);

Running query in 'postgresql://localhost:5432/gnis'

1174 rows affected.

feature_id,feature_name,feature_class,state_alpha,state_numeric,county_name,county_numeric,primary_lat_dms,prim_long_dms,prim_lat_dec,prim_long_dec,source_lat_dms,source_long_dms,source_lat_dec,source_long_dec,elev_in_m,elev_in_ft,map_name,date_created,date_edited
1230,Belmont Mountains,Range,AZ,4,Maricopa,13.0,333832N,1125404W,33.642258,-112.9010129,None,None,None,None,931.0,3054.0,Belmont Mountain,02/08/1980,None
2750,Chandler Springs,Spring,AZ,4,Navajo,17.0,352236N,1102831W,35.3766788,-110.4754096,None,None,None,None,1685.0,5528.0,Shonto Butte,02/08/1980,None
5427,Gunsight Canyon,Valley,AZ,4,Mohave,15.0,344716N,1133708W,34.7877893,-113.6188253,344718N,1134658W,34.7883432,-113.7827192,642.0,2106.0,Tule Wash,02/08/1980,None
13547,West Well,Well,AZ,4,Navajo,17.0,350208N,1102857W,35.0355772,-110.4826281,None,None,None,None,1517.0,4977.0,Humpy Camp Well,02/08/1980,03/27/2018
13609,White Cone,Summit,AZ,4,Apache,1.0,361115N,1090452W,36.1875054,-109.0812086,None,None,None,None,2569.0,8428.0,Upper Wheatfields,02/08/1980,None
13727,Whittier Elementary School,School,AZ,4,Maricopa,13.0,332812N,1120256W,33.4700933,-112.0488972,None,None,None,None,337.0,1106.0,Phoenix,02/08/1980,09/17/2018
15418,Kinnison Wash,Arroyo,AZ,4,Pima,19.0,321112N,1104908W,32.1867438,-110.818975,320757N,1104908W,32.1325,-110.8188889,822.0,2697.0,Tucson East,06/27/1984,None
27595,China Spring Creek,Stream,AZ,4,Gila,7.0,335506N,1105416W,33.9183801,-110.9045635,335617N,1105131W,33.9381028,-110.8587295,1253.0,4111.0,McFadden Peak,02/08/1980,None
29236,Gabarina Hill,Summit,AZ,4,Yavapai,25.0,342932N,1123513W,34.4922062,-112.5869989,None,None,None,None,1689.0,5541.0,Wilhoit,02/08/1980,03/10/2021
29322,Gil Water Tank,Reservoir,AZ,4,Pima,19.0,312705N,1112622W,31.451481,-111.4395465,None,None,None,None,1109.0,3638.0,Cumero Mountain,02/08/1980,None


# [At Home] Numerical Granularity

In [19]:
%sql SELECT elev_in_m FROM National LIMIT 2;

Running query in 'postgresql://localhost:5432/gnis'

2 rows affected.

elev_in_m
931.0
2707.0


In [20]:
%%sql
SELECT elev_in_m, 
    (elev_in_m / 100)::INTEGER AS quantized,
    ((elev_in_m / 100)::INTEGER) * 100 AS round_to_100,
    SUBSTRING(elev_in_m::TEXT, 1, 2),
    CONCAT(SUBSTRING(elev_in_m::TEXT, 1, 2), '00') AS substring2
FROM National
LIMIT 5;

Running query in 'postgresql://localhost:5432/gnis'

5 rows affected.

elev_in_m,quantized,round_to_100,substring,substring2
931.0,9,900,93,9300
2707.0,27,2700,27,2700
275.0,3,300,27,2700
1685.0,17,1700,16,1600
1354.0,14,1400,13,1300


In [21]:
%config SqlMagic.named_parameters=True

/srv/conda/envs/notebook/lib/python3.11/site-packages/sql/traits.py:20: FutureWarning: named_parameters: boolean values are now deprecated. Value True will be treated as "enabled". 
Please use a valid option: "warn", "enabled", or "disabled". 
For more information, see the docs: https://jupysql.ploomber.io/en/latest/api/configuration.html#named-parameters
  warnings.warn(


In [22]:
right_shift = '>>'
left_shift = '<<'

In [23]:
%%sql
/* Since jupysql does not like bitshifts, we can fake it with string interoplation. */
SELECT elev_in_m,
    (16::INTEGER::BIT(12)) AS bit12,
    (16::INTEGER::BIT(12)) {{left_shift}} 3
FROM national
LIMIT 5;

Running query in 'postgresql://localhost:5432/gnis'

5 rows affected.

elev_in_m,bit12,?column?
931.0,000000010000,000010000000
2707.0,000000010000,000010000000
275.0,000000010000,000010000000
1685.0,000000010000,000010000000
1354.0,000000010000,000010000000


In [24]:
%%sql
EXPLAIN (verbose true)
WITH shifts AS (
    SELECT elev_in_m,
       (elev_in_m::integer::bit(12)) AS bit12,
       (elev_in_m::integer::bit(12) {{right_shift}} 8) AS rightshifted,
       ((elev_in_m::integer::bit(12) {{right_shift}} 8) {{left_shift}} 8)::integer AS round_to_256,
       ((elev_in_m::integer::bit(12) {{right_shift}} 8) {{left_shift}} 8)::integer % 256 AS test
  FROM national
)
SELECT COUNT(DISTINCT elev_in_m) AS elevation_meters_count,
       COUNT(DISTINCT bit12) AS bit12_count,
       COUNT(DISTINCT rightshifted) AS rightshift_count,
       COUNT(DISTINCT round_to_256) AS rounded_count
  FROM shifts;

Running query in 'postgresql://localhost:5432/gnis'

4 rows affected.

QUERY PLAN
Aggregate (cost=799.99..800.00 rows=1 width=32)
"Output: count(DISTINCT ""national"".elev_in_m), count(DISTINCT ((""national"".elev_in_m)::integer)::bit(12)), count(DISTINCT (((""national"".elev_in_m)::integer)::bit(12) >> 8)), count(DISTINCT (((((""national"".elev_in_m)::integer)::bit(12) >> 8) << 8))::integer)"
"-> Seq Scan on public.""national"" (cost=0.00..396.33 rows=11533 width=8)"
"Output: ""national"".feature_id, ""national"".feature_name, ""national"".feature_class, ""national"".state_alpha, ""national"".state_numeric, ""national"".county_name, ""national"".county_numeric, ""national"".primary_lat_dms, ""national"".prim_long_dms, ""national"".prim_lat_dec, ""national"".prim_long_dec, ""national"".source_lat_dms, ""national"".source_long_dms, ""national"".source_lat_dec, ""national"".source_long_dec, ""national"".elev_in_m, ""national"".elev_in_ft, ""national"".map_name, ""national"".date_created, ""national"".date_edited"


# [At Home] Demo 1: Roll-up / Drill-down Practice

Let's start with county-level data on elevations:

In [25]:
%%sql
SELECT state_numeric, county_numeric,
       avg(elev_in_m),
       stddev(elev_in_m), count(*)
FROM national TABLESAMPLE BERNOULLI(10)
GROUP BY state_numeric, county_numeric;

Running query in 'postgresql://localhost:5432/gnis'

852 rows affected.

state_numeric,county_numeric,avg,stddev,count
56,35.0,3323.0,274.35743110038044,2
1,33.0,153.5,10.606601717798213,2
53,73.0,103.0,None,1
46,47.0,990.0,None,1
29,169.0,241.0,None,1
49,49.0,1646.0,None,1
29,31.0,127.5,19.091883092036785,2
20,55.0,863.0,None,1
24,19.0,0.0,None,1
22,107.0,22.0,4.242640687119285,2


**Roll up** to state level.
* We save the view as `state_elevations` for later...

In [ ]:
%%sql
DROP VIEW IF EXISTS state_elevations;

CREATE VIEW state_elevations AS (
    SELECT state_numeric,
       avg(elev_in_m),
       stddev(elev_in_m), count(*)
    FROM national
    GROUP BY state_numeric
);

In [ ]:
%sql SELECT * FROM state_elevations;

**Drill down** to include feature class.

In [ ]:
%%sql
SELECT state_numeric, feature_class,
       avg(elev_in_m),
       stddev(elev_in_m), count(*)
FROM national TABLESAMPLE Bernoulli(10)
GROUP BY state_numeric, feature_class
ORDER BY count(*) DESC;

# Demo 2: Connections to Statistics

## [At home] Roll up with marginal distributions

In [ ]:
%%sql
SELECT state_numeric,
       AVG(elev_in_m),
       STDDEV(elev_in_m), COUNT(*),
       SUM(COUNT(*)) OVER () AS total,
       COUNT(*)/SUM(COUNT(*)) OVER () AS marginal
FROM national TABLESAMPLE Bernoulli(.07)
GROUP BY state_numeric;

In [ ]:
%%sql
SELECT COUNT(DISTINCT county_numeric) FROM national;

## Drill down with normally-distributed elevations:

We will start with a view that is at the state granularity, and then go from there. (Same as "At Home" exercise above)

In [ ]:
%%sql
DROP VIEW IF EXISTS state_elevations;

CREATE VIEW state_elevations AS (
    SELECT state_numeric,
       avg(elev_in_m),
       stddev(elev_in_m), count(*)
    FROM national
    GROUP BY state_numeric
);

Start with the `state_elevations` view from earlier:

In [ ]:
%sql SELECT * FROM state_elevations;

The `fips_counties` relation has all counties, including those not in `national`:

In [ ]:
%sql SELECT * FROM fips_counties WHERE state_numeric = 6 LIMIT 10;

If we wanted to **drill down** to the FIPS counties, we'd need to simulate an elevation for those counties that don't exist in `national`.

Here's the first step in that process, which creates a simulated value for *every* county in `fips_counties`.
* The value is simulated from a normal distribution using that state's elevation statistics (average, standard deviation).
* Just like a Python package, we need to import `tablefunc` in order to use the `normal_rand` function.

A few other details:
* Need `WITH ORDINALITY` to add row numbers to each sample generated via `normal_rand`
* We join on these row numbers to ensure that each county gets a unique sampled value

In [ ]:
%sql CREATE EXTENSION IF NOT EXISTS tablefunc;

Adding row numbers

In [ ]:
%%sql 
SELECT s.state_numeric, f.fips as county_numeric, s.avg, s.stddev, s.count, 
    rank() OVER (PARTITION BY s.state_numeric ORDER BY f.fips) as rk
FROM state_elevations s, fips_counties f
WHERE s.state_numeric = f.state_numeric

In [ ]:
%%sql
WITH state_cty AS
(SELECT s.state_numeric, f.fips as county_numeric, s.avg, s.stddev, s.count, rank() OVER (PARTITION BY s.state_numeric ORDER BY f.fips) as rk
  FROM state_elevations s, fips_counties f
  WHERE s.state_numeric = f.state_numeric
)
SELECT s.*,
       n.rn AS sampled_rk,
       n.n AS elev_in_m,
       true as elev_in_m_sim -- user-facing flag
  FROM state_cty s,
       LATERAL (SELECT n, rn FROM normal_rand(CAST(s.count AS INTEGER), s.avg, s.stddev) WITH ORDINALITY AS n(n, rn)) n
  WHERE s.rk = n.rn
LIMIT 10;

In [ ]:
%%sql
SELECT n, rn FROM normal_rand(1000, 3, 0.5) WITH ORDINALITY AS n(n, rn)

# [At home] Assembling an Explicit Hierarchy

In [ ]:
## we'll use the Lahman baseball database in our initial examples today.
## replace the database connection with a database of your own!
%reload_ext sql
%sql postgresql://localhost:5432/baseball

Two relations have the pieces of the hierarchy we want:

In [ ]:
%sql SELECT * FROM Appearances WHERE year_id > 1970 LIMIT 2;

In [ ]:
%sql SELECT * FROM Teams LIMIT 1;

Let's join these two to make our hierarchy! Which way should we make this?

In [ ]:
%%sql
SELECT a.player_id, a.team_id, t.div_id, a.*
FROM Appearances a
NATURAL JOIN Teams t
WHERE a.year_id = 2015
LIMIT 100;

In [ ]:
%%sql
CREATE OR REPLACE VIEW bball_tree AS (
    SELECT DISTINCT
        a.player_id, a.team_id, t.div_id,
        a.lg_id, a.year_id
    FROM appearances a
    NATURAL JOIN teams t
);

In [ ]:
%sql SELECT * FROM bball_tree WHERE div_id IS NOT NULL LIMIT 25;

### Revisiting the Home Run Query

Recall our old home run query:

In [ ]:
%%sql
SELECT name_first, name_last, year_id,
       MIN(hr), MAX(hr), AVG(hr), STDDEV(hr), SUM(hr)
FROM batting b, people p
WHERE b.player_id = p.player_id
GROUP BY name_last, name_first, year_id
ORDER BY max DESC
LIMIT 10;

Set up for roll up/drill down on `bball_tree` hierarchy.
* Join each (raw) person with the associated `bball_tree` entry by `(playerid, yearid)` in a CTE
* Use this result for roll-up and drill-down.

(blank space before we get to the next exercise....)
<br/><br/><br/><br/><br/>
<br/><br/><br/><br/><br/>
<br/><br/><br/><br/><br/>
<br/><br/><br/><br/><br/>

In [ ]:
%%sql
WITH batting_tree AS (
    SELECT b.*, t.div_id
    FROM batting b, bball_tree t
    WHERE b.player_id = t.player_id
      AND b.year_id = t.year_id
)
SELECT name_first, name_last,
       bt.team_id, bt.lg_id, bt.div_id, bt.year_id,
       MIN(hr), MAX(hr), AVG(hr), STDDEV(hr), SUM(hr)
FROM batting_tree bt, people p
WHERE bt.player_id = p.player_id
GROUP BY bt.player_id, bt.team_id, bt.lg_id, bt.div_id, bt.year_id, name_last, name_first
ORDER BY max DESC
LIMIT 10;
